# SignBridge — Model Training (Colab GPU)
Run cells top to bottom. Change `ARCH` to switch between models.

In [ ]:
# 1. Install dependencies
!pip install wandb --quiet

In [ ]:
# 2. Mount Google Drive (save checkpoints here so they survive session end)
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/SignBridge/checkpoints'
import os; os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# 3. Clone your repo and upload landmarks
!git clone https://github.com/HaideyAli/SignSync.git /content/SignSync
%cd /content/SignSync
# Then upload your data/landmarks/ folder and data/labels.json to /content/SignSync/data/

In [ ]:
# 4. Log in to wandb
import wandb
wandb.login()

In [ ]:
# 5. Config — change ARCH to 'lstm' for baseline or 'transformer' for primary model
ARCH   = 'transformer'
EPOCHS = 100
LR     = 1e-3
BATCH  = 32

In [ ]:
# 6. Training (copy of src/train.py)
import sys, math, torch, torch.nn as nn
from pathlib import Path
sys.path.insert(0, '/content/SignSync/src')
from dataset import create_dataloaders
from model import build_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for seqs, labels in loader:
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            logits = model(seqs)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

train_loader, val_loader, label_map = create_dataloaders(
    landmarks_dir='/content/SignSync/data/landmarks',
    labels_path='/content/SignSync/data/labels.json',
    batch_size=BATCH
)
model     = build_model(ARCH).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

wandb.init(project='SignBridge', name=ARCH, config={'arch': ARCH, 'epochs': EPOCHS, 'lr': LR})

best_acc, no_improve = 0.0, 0
ckpt_path = f'{CHECKPOINT_DIR}/best_model_{ARCH}.pth'

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)
    scheduler.step(val_loss)
    saved = val_acc > best_acc
    if saved:
        torch.save({'model_state': model.state_dict(), 'arch': ARCH}, ckpt_path)
        best_acc = val_acc
        no_improve = 0
    else:
        no_improve += 1
    print(f'Epoch {epoch:3d} | train {train_acc:.3f} | val {val_acc:.3f}' + (' *' if saved else ''))
    wandb.log({'train_loss': train_loss, 'train_acc': train_acc,
               'val_loss': val_loss, 'val_acc': val_acc, 'best_val_acc': best_acc})
    if no_improve >= 10:
        print('Early stopping.'); break

print(f'Best val accuracy: {best_acc:.3f} — saved to {ckpt_path}')
wandb.finish()

In [ ]:
# 7. Download checkpoint to your machine
from google.colab import files
files.download(ckpt_path)